In [110]:
import math
class Value:
    def __init__(self, data, parents=()):
        self.data = data
        self.parents = parents
        self.grad = 0
        self._backward = lambda: None
    
    def __add__(self, val):
        if not isinstance(val, Value):
            val = Value(val)
        out = Value(self.data + val.data, parents=(self,val))

        def _backward():
            self.grad += 1 * out.grad
            val.grad += 1 * out.grad

        out._backward = _backward

        return out
    
    def __radd__(self, val):
        return self.__add__(val)
    
    def __sub__(self, val):
        if not isinstance(val, Value):
            val = Value(val)
        out = Value(self.data - val.data, parents=(self,val))

        def _backward():
            self.grad += 1 * out.grad
            val.grad += -1 * out.grad

        out._backward = _backward
        return out

    def __rsub__(self, val):
        return val + (-self)
        
    
    def __mul__(self, val):
        if not isinstance(val, Value):
            val = Value(val)
        out = Value(self.data * val.data, parents=(self, val))

        def _backward():
            self.grad += val.data * out.grad
            val.grad += self.data * out.grad

        out._backward = _backward
        return out
    
    def __rmul__(self, val):
        return self.__mul__(val)
    
    def __truediv__(self, val):
        if not isinstance(val, Value):
            val = Value(val)
        out = Value(self.data / val.data, parents=(self,val))
    

        def _backward():
            self.grad += (1 / val.data) * out.grad
            val.grad += (-self.data / (val.data**2)) * out.grad
        
        out._backward = _backward

        return out

    def __rtruediv__(self, val):
        return val * (self ** -1)

    def __pow__(self, val):
        if not isinstance(val, Value):
            out = Value(self.data ** val, parents = (self,))
            def _backward():
                self.grad +=  (val * self.data ** (val-1)) * out.grad
        else:
            out = Value(self.data ** val.data, parents=(self,val))
            def _backward():
                self.grad +=  (val.data * self.data ** (val.data-1)) * out.grad
                val.grad += math.log(self.data) * out.data * out.grad
        
        out._backward = _backward
        return out
        
    def __neg__(self):
        out = Value(self.data * -1, parents = (self,))
        def _backward():
                self.grad += -out.grad
        out._backward = _backward
        return out
    
    def relu(self):
        out = Value(max(0, self.data), parents = (self,))

        if out.data == 0:
            def _backward():
                self.grad += 0.0
        else:
            def _backward():
                self.grad += out.grad
        
        out._backward = _backward

        return out
    
    def backward(self):
        visited = set()
        topo = []

        def build_topo(val):
            if val not in visited:
                visited.add(val)
                for parent in val.parents:
                    build_topo(parent)
                topo.append(val)
        
        build_topo(self)
        self.grad = 1.0
        
        for val in reversed(topo):
            val._backward()

In [111]:
class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

    def parameters(self):
            return []

In [116]:
import random
class Neuron(Module):
    def __init__(self, nin, nonlin):
        scale = nin ** -0.5 # scaled to avoid exploding/shrinking gradients
        self.w = [Value(random.uniform(-1, 1) * scale) for n in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, inputs):
        s = self.b
        for wi, xi in zip(self.w, inputs):
            s = s + wi*xi
        return s.relu() if self.nonlin else s
    
    def parameters(self):
        return self.w + [self.b]

In [117]:
class Layer(Module):
    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]
    
    def __call__(self, inputs):
        outs = [n(inputs) for n in self.neurons]
        return outs
    
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

In [118]:
class MLP(Module):
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [
            Layer(sz[i], sz[i+1], nonlin=(i != len(nouts) - 1))
            for i in range(len(nouts))
        ]
    
    def __call__(self, inputs):
        for layer in self.layers:
            inputs = layer(inputs)
        return inputs
    
    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]

In [144]:
# Inputs: 4 combinations of binary inputs
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
# Target outputs (+1.0 or -1.0)
ys = [1.0, -1.0, -1.0, 1.0]

random.seed(42)
model = MLP(3, [4, 4, 1])

lr = 0.05
for epoch in range(100):
    y_pred = [model(x) for x in xs]
    loss = sum((yout[0] - ygt)**2 for ygt, yout in zip(ys, y_pred))
    
    model.zero_grad()
    loss.backward()
    
    for p in model.parameters():
        p.data -= lr * p.grad

    if epoch % 5 == 0:
        print(f"Epoch {epoch:2d} | Loss: {loss.data:.6f}")

Epoch  0 | Loss: 4.097524
Epoch  5 | Loss: 3.749151
Epoch 10 | Loss: 3.330156
Epoch 15 | Loss: 2.215439
Epoch 20 | Loss: 0.967235
Epoch 25 | Loss: 0.132330
Epoch 30 | Loss: 0.020822
Epoch 35 | Loss: 0.009848
Epoch 40 | Loss: 0.006887
Epoch 45 | Loss: 0.005034
Epoch 50 | Loss: 0.003756
Epoch 55 | Loss: 0.002847
Epoch 60 | Loss: 0.002186
Epoch 65 | Loss: 0.001697
Epoch 70 | Loss: 0.001329
Epoch 75 | Loss: 0.001050
Epoch 80 | Loss: 0.000834
Epoch 85 | Loss: 0.000666
Epoch 90 | Loss: 0.000534
Epoch 95 | Loss: 0.000429
